In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName('agg').getOrCreate()

In [10]:
bnb = spark.read.csv('file:///Users/User/Downloads/air_bnb.csv',header=True)

In [11]:
bnb.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: string (nullable = true)
 |-- minimum_nights: string (nullable = true)
 |-- number_of_reviews: string (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: string (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: string (nullable = true)



In [12]:
from pyspark.sql.types import *

In [13]:
# col.cast(type_function): col의 type을 변경하여 반환
# ex> bnb.price.cast(IntegerType())
# bnb의 col을 아래와 같이 type 변경
# price: IntegerType()
# minimum_nights: IntegerType()
# number_of_reviews: IntegerType()
# reviews_per_month: FloatType()
# alailability_365: IntegerType()
bnb = bnb.withColumn('price',bnb.price.cast(IntegerType()))\
.withColumn('minimum_nights',bnb.minimum_nights.cast(IntegerType()))\
.withColumn('number_of_reviews',bnb.number_of_reviews.cast(IntegerType()))\
.withColumn('reviews_per_month',bnb.reviews_per_month.cast(FloatType()))\
.withColumn('availability_365',bnb.availability_365.cast(IntegerType()))

In [14]:
bnb.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- host_id: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- neighbourhood_group: string (nullable = true)
 |-- neighbourhood: string (nullable = true)
 |-- latitude: string (nullable = true)
 |-- longitude: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- last_review: string (nullable = true)
 |-- reviews_per_month: float (nullable = true)
 |-- calculated_host_listings_count: string (nullable = true)
 |-- availability_365: integer (nullable = true)



## groupBy
- 지정된 열의 값을 기준으로 grouping함
- 지정된 열의 값은 일반적으로 **categorical data**(범주형 데이터)<br/>
  (categorical data: 한정적인 수의 값을 갖는 데이터 (ex> 남/여, 10대/20대/30대/.../90대)
- groupBy를 이용하여 grouping을 할 때, 반드시 해당 group의 집계된 값을 배정하기 위한 통계함수도 함께 적용되어야 함

In [19]:
# neighbourhood_group 각 group의 데이터 수(count)
print(type(bnb.groupBy('neighbourhood_group'))) 
# groupBy의 결과는 GroupedData, DataFrame이 아님
# GroupedData란 지정된 열의 값에 따라 grouping된 DataFrame의 집합
# GroupedData에 통계함수를 적용하여 각 DataFrame의 집계값을 결정한 뒤, 결과를 DataFrame으로 반환
bnb.groupBy('neighbourhood_group').count().show(10)

<class 'pyspark.sql.group.GroupedData'>
+-------------------+-----+
|neighbourhood_group|count|
+-------------------+-----+
|         Douglaston|    1|
|             Queens| 5521|
|              Nadia|    1|
|            Midtown|    4|
|     Hell's Kitchen|    6|
|  Greenwich Village|    2|
|       Clinton Hill|    1|
|   Ditmars Steinway|    2|
|           Longwood|    2|
|        Little Neck|    1|
+-------------------+-----+
only showing top 10 rows



In [22]:
# neighbourhood_group 각 group의 price 평균값
bnb.groupBy('neighbourhood_group').mean('price').show(10)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
|  Greenwich Village|              55.5|
|       Clinton Hill|              14.0|
|   Ditmars Steinway|               3.5|
|           Longwood|               5.0|
|        Little Neck|               1.0|
+-------------------+------------------+
only showing top 10 rows



## agg
- **GroupedData.agg({col: 통계함수})** or **GroupedData.agg(통계함수(col))**
- 복수의 column과 통계함수를 적용할 수 있음

In [25]:
from pyspark.sql.functions import *

In [28]:
bnb.groupBy('neighbourhood_group').agg(mean(bnb.price)).show(10)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
|  Greenwich Village|              55.5|
|       Clinton Hill|              14.0|
|   Ditmars Steinway|               3.5|
|           Longwood|               5.0|
|        Little Neck|               1.0|
+-------------------+------------------+
only showing top 10 rows



In [29]:
bnb.groupBy('neighbourhood_group').agg({'price':'mean'}).show(10)

+-------------------+------------------+
|neighbourhood_group|        avg(price)|
+-------------------+------------------+
|         Douglaston|               1.0|
|             Queens| 99.60460061583046|
|              Nadia|              NULL|
|            Midtown|               9.0|
|     Hell's Kitchen|1.3333333333333333|
|  Greenwich Village|              55.5|
|       Clinton Hill|              14.0|
|   Ditmars Steinway|               3.5|
|           Longwood|               5.0|
|        Little Neck|               1.0|
+-------------------+------------------+
only showing top 10 rows



In [30]:
# neighbourhood_group 각 group의 price 평균값(mean)과 number_of_reviews의 최대값(max)
bnb.groupBy('neighbourhood_group').agg({'price':'mean','number_of_reviews':'max'}).show(10)

+-------------------+------------------+----------------------+
|neighbourhood_group|        avg(price)|max(number_of_reviews)|
+-------------------+------------------+----------------------+
|         Douglaston|               1.0|                  NULL|
|             Queens| 99.60460061583046|                   629|
|              Nadia|              NULL|                    30|
|            Midtown|               9.0|                  NULL|
|     Hell's Kitchen|1.3333333333333333|                  NULL|
|  Greenwich Village|              55.5|                  NULL|
|       Clinton Hill|              14.0|                  NULL|
|   Ditmars Steinway|               3.5|                  NULL|
|           Longwood|               5.0|                  NULL|
|        Little Neck|               1.0|                  NULL|
+-------------------+------------------+----------------------+
only showing top 10 rows



In [31]:
bnb.groupBy('neighbourhood_group').agg(mean('price'),max('number_of_reviews')).show(10)

+-------------------+------------------+----------------------+
|neighbourhood_group|        avg(price)|max(number_of_reviews)|
+-------------------+------------------+----------------------+
|         Douglaston|               1.0|                  NULL|
|             Queens| 99.60460061583046|                   629|
|              Nadia|              NULL|                    30|
|            Midtown|               9.0|                  NULL|
|     Hell's Kitchen|1.3333333333333333|                  NULL|
|  Greenwich Village|              55.5|                  NULL|
|       Clinton Hill|              14.0|                  NULL|
|   Ditmars Steinway|               3.5|                  NULL|
|           Longwood|               5.0|                  NULL|
|        Little Neck|               1.0|                  NULL|
+-------------------+------------------+----------------------+
only showing top 10 rows



In [32]:
bnb.groupBy('neighbourhood_group').agg(mean('price').alias('mean_price'),\
                                       max('number_of_reviews').alias('max_reviews')).show(10)

+-------------------+------------------+-----------+
|neighbourhood_group|        mean_price|max_reviews|
+-------------------+------------------+-----------+
|         Douglaston|               1.0|       NULL|
|             Queens| 99.60460061583046|        629|
|              Nadia|              NULL|         30|
|            Midtown|               9.0|       NULL|
|     Hell's Kitchen|1.3333333333333333|       NULL|
|  Greenwich Village|              55.5|       NULL|
|       Clinton Hill|              14.0|       NULL|
|   Ditmars Steinway|               3.5|       NULL|
|           Longwood|               5.0|       NULL|
|        Little Neck|               1.0|       NULL|
+-------------------+------------------+-----------+
only showing top 10 rows



In [37]:
bnb.select('price','number_of_reviews').summary().show()

+-------+------------------+-----------------+
|summary|             price|number_of_reviews|
+-------+------------------+-----------------+
|  count|             48312|            48176|
|   mean| 152.3312427554231|23.26349219528396|
| stddev|239.34030523202495|44.59808062733653|
|    min|               -74|                0|
|    25%|                69|                1|
|    50%|               105|                5|
|    75%|               175|               23|
|    max|             10000|              629|
+-------+------------------+-----------------+



In [41]:
bnb.groupBy('reviews_per_month').count().show()
# groupBy를 통한 grouping은 categorical data를 이용하며, 일반적으로 연속적인 값을 갖는 data로는 grouping을 하지 않는다.

+-----------------+-----+
|reviews_per_month|count|
+-----------------+-----+
|             3.32|   24|
|             5.85|    3|
|             4.96|   11|
|             0.34|  165|
|              6.9|    3|
|             0.42|  225|
|             1.63|   47|
|             3.36|   23|
|             5.44|    6|
|             9.41|    2|
|             0.73|  110|
|             0.15|  367|
|             3.22|   29|
|            11.91|    1|
|             8.87|    4|
|             6.94|    2|
|             4.04|   22|
|            11.17|    1|
|             1.13|   77|
|             4.03|   15|
+-----------------+-----+
only showing top 20 rows



## pivot

In [42]:
sales = spark.read.csv('file:///Users/User/Downloads/supermarket_sales1.csv',\
                      inferSchema=True, header=True)
sales.printSchema()

root
 |-- Invoice ID: string (nullable = true)
 |-- Branch: string (nullable = true)
 |-- City: string (nullable = true)
 |-- Customer type: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Product line: string (nullable = true)
 |-- Unit price: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Tax 5%: double (nullable = true)
 |-- Total: double (nullable = true)
 |-- Date: string (nullable = true)
 |-- Time: timestamp (nullable = true)
 |-- Payment: string (nullable = true)
 |-- cogs: double (nullable = true)
 |-- gross margin percentage: double (nullable = true)
 |-- gross income: double (nullable = true)
 |-- Rating: double (nullable = true)



In [44]:
# 각 도시(city)에 대해서 Ewallet과 Credit card로 지불한 사람의 수(count)를 집계
# 1. 각 도시에 대한 수를 확인해야 하므로, city를 기준으로 grouping
# 2. GroupedData.pivot(col,[val1,val2]).통계함수: 
#     - grouping된 데이터에 대해 특정 column이 지정된 값을 갖는 데이터를 기준으로 집계
sales.groupBy('city').pivot('Payment',['Ewallet','Credit card']).count().show(10)

+---------+-------+-----------+
|     city|Ewallet|Credit card|
+---------+-------+-----------+
|Naypyitaw|    106|         98|
| Mandalay|    113|        109|
|   Yangon|    126|        104|
+---------+-------+-----------+



In [ ]:
# 각 성별에 대해 Ewallet과 Credit card로 지불한 사람들의 Total값의 최대, 최소값
# 최대/최소값을 포함한 열의 이름은 각각 max_Total, min_Total로 하시오.
# -> Ewallet과 Credit card로 지불한 남자와 여자의 최대 금액과 최소 금액
